# Intelligent Agents: Reflex-Based Agents for GridHunt

Student Name: [Add your name]

I have used the following AI tools: [list tools]

I understand that my submission needs to be my own work: [your initials]

## Learning Outcomes

* Apply core AI concepts by implementing the agent function for a simple and model-based reflex agents that respond to environmental percepts.
* Practice how the environment and the agent function interact.
* Analyze agent performance through controlled experiments across different environment configurations.

## Instructions

Total Points: Undergrads 10

Complete this notebook. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook.
### AI Use

Here are some guidelines that will make it easier for you:

* __Don't:__ Rely on AI auto completion. You will waste a lot of time trying to figure out how the suggested code relates to what we do in class. Turn off AI code completion (e.g., Copilot) in your IDE.
* __Don't:__ Do not submit code/text that you do not understand or have not checked to make sure that it is complete and correct.
* __Do:__ Use AI for debugging and letting it explain code and concepts from class.

## Introduction

![GridHunt Image](gridhunt.png)

Gridhunt is a small educational game that helps you practice implementing simple agent functions. It is a modified Python reimplementation of Gridhunt2.

The objective of gridhunt is to implement an intelligent agent, the hunter, who can catch a monster faster than all other hunters. Gridhunt uses a $n \times n$ arena consisting of a grid of tiles. Gridhunt is a turn-based game, and the hunter and the monster move on this grid. A hunter catches the monster if she/he moves on the same square the monster currently occupies. If the monster survives the maximum number of steps for the game, then the monster wins.

## PEAS Description of Gridhunt

__Performance Measure:__ The performance is measured as the number of steps the hunter uses to catch the monster. Catching the monster means to be on the same square.

__Environment:__ An arena with $n \times n$ squares. At the beginning the monster and the hunter are randomly placed in the arena environment. The hunter and the monster
    can move around, but cannot leave the arena.

__Actuators:__ The agent can move to an adjacent square using actions "north", "east", "west", and "south", or teleport which will move the agent
    to a random square in the arena.

__Sensors:__ The agent can always see its location in the arena but only gets the monster location when it stops and listens.

## The Arena Environment

The environment manages the location of the agents (hunter and monster). It initially places them in a random location and then provides them in every step with percepts (the location of the hunter and the monster) and asks them for their action.

__A Note on positions:__
The arena is implemented as an array with row and column indices representing the position.
Positions are stored in a numpy array where `pos[0]` is the row index and `pos[1]` is the column index in the arena. North means up in this array, that is the row index gets smaller when going north. Remember indices in Python start with 0 and go to `n-1`.

In [89]:
import numpy as np
from IPython.display import clear_output
from time import sleep

def arena_environment(hunter_agent_function,
                      monster_agent_function,
                      n = 30, max_steps = 100,
                      visualize = False, animation = False):
    """
    Simulate an arena where a hunter agent tries to catch a monster agent.

    Parameters:
    hunter_agent_function (function): A function that takes the current positions of the hunter and monster
                                      and returns the next move for the hunter ('north', 'east', 'west', 'south', 'teleport').
    monster_agent_function (function): A function that takes the current positions of the hunter and monster
                                       and returns the next move for the monster (or 'stay' to remain in place).
    n (int): The size of the arena (n x n grid).
    max_steps (int): The maximum number of steps to simulate.
    visualize (bool): Whether to visualize the arena.
    animation (bool): Whether to animate the visualization with a delay.

    Returns:
    int: The number of steps taken for the hunter to catch the monster or np.nan (not a number) if not caught.
    """

    # Initialize positions
    monster_pos = np.random.randint(0, n-1, size=2)
    hunter_pos = np.random.randint(0, n-1, size=2)

    def move(action, position):
        """calculate new position for the agent."""
        if action == 'north':
            position[0] -= 1
        elif action == 'south':
            position[0] += 1
        elif action == 'west':
            position[1] -= 1
        elif action == 'east':
            position[1] += 1
        else:
            raise ValueError("Invalid action.")

        # Ensure position stays within bounds
        position = np.clip(position, 0, n-1)
        return position

    def print_arena():

        if animation:
            sleep(1)
            clear_output(wait=True)
        print(f"Step {step+1}: Hunter is at '{hunter_pos}'; Monster is at '{monster_pos}'")
        arena = np.full((n, n), '.', dtype=str)
        arena[monster_pos[0], monster_pos[1]] = 'M'
        arena[hunter_pos[0], hunter_pos[1]] = 'H'
        if np.array_equal(hunter_pos, monster_pos):
            arena[monster_pos[0], monster_pos[1]] = '*'
        print("\n".join("".join(row) for row in arena))

    # run the environment
    hunter_action = None  # Initialize hunter_action to None for the first step
    for step in range(max_steps):
        if visualize:
            print_arena()

        # Check if hunter has caught the monster
        if np.array_equal(hunter_pos, monster_pos):
            if visualize:
                print(f"Hunter caught the monster in {step} steps!")
            return step

        # Get next move from monster agent function
        monster_action = monster_agent_function(hunter_pos, monster_pos)

        if monster_action != 'stay':
            monster_pos = move(monster_action, monster_pos)

        # Get next move from hunter agent function. The monster's position is only provided
        # if the hunter chose 'listen' as the previous action.
        if hunter_action == 'listen':
            monster_pos_instrument = monster_pos
        else:
            monster_pos_instrument = None

        hunter_action = hunter_agent_function(hunter_pos, monster_pos_instrument)

        if hunter_action == 'listen':
            pass
        elif hunter_action == 'teleport':
            hunter_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_pos = move(hunter_action, hunter_pos)

        # print the agents' actions
        if visualize:
            print(f"Hunter chose action '{hunter_action}'")
            print(f"Monster chose action '{monster_action}'")
            print("\n")

    # the hunter failed to catch the monster within max_steps
    if visualize:
        print(f"Hunter failed to catch the monster in {max_steps} steps.")

    return np.nan

I implement the monster here as a simple agent that moves around randomly, but mostly stays in its place. You can use AI to get a detailed explanation of how the following code works.

In [90]:
monster_actions = ["north", "east", "west", "south", "stay"]

def monster_agent_function_simple(hunter_pos, monster_pos):
    return np.random.choice(monster_actions, p=[0.125, 0.125, 0.125, 0.125, 0.5])

# The Hunter Agent

Your job is to implement a simple-reflex hunter agent that can catch the monster. Remember, your agent is implemented as an agent function that get percepts and needs to return a valid action. The actions are:

In [91]:
actions = ["north", "east", "west", "south", "teleport", "listen"]

## A Simple Example Implementation

Here is a very simple hunter that just runs around randomly and hopes that it bumps into the monster.

In [92]:
def simple_randomized_hunter_agent_function(hunter_location, monster_location):
    return np.random.choice(actions)

Ask the agent for an action.

In [93]:
simple_randomized_hunter_agent_function([0,0], None)

np.str_('north')

## Experimenting With the Agent

We can place the monster and the hunter into the environment and run a simulation by calling the environment function.

In [94]:
arena_environment(simple_randomized_hunter_agent_function,
                  monster_agent_function_simple,
                  n=5, max_steps=10, visualize=True, animation=False)

Step 1: Hunter is at '[1 0]'; Monster is at '[2 1]'
.....
H....
.M...
.....
.....
Hunter chose action 'east'
Monster chose action 'stay'


Step 2: Hunter is at '[1 1]'; Monster is at '[2 1]'
.....
.H...
.M...
.....
.....
Hunter chose action 'north'
Monster chose action 'west'


Step 3: Hunter is at '[0 1]'; Monster is at '[2 0]'
.H...
.....
M....
.....
.....
Hunter chose action 'listen'
Monster chose action 'stay'


Step 4: Hunter is at '[0 1]'; Monster is at '[2 0]'
.H...
.....
M....
.....
.....
Hunter chose action 'north'
Monster chose action 'south'


Step 5: Hunter is at '[0 1]'; Monster is at '[3 0]'
.H...
.....
.....
M....
.....
Hunter chose action 'east'
Monster chose action 'east'


Step 6: Hunter is at '[0 2]'; Monster is at '[3 1]'
..H..
.....
.....
.M...
.....
Hunter chose action 'listen'
Monster chose action 'stay'


Step 7: Hunter is at '[0 2]'; Monster is at '[3 1]'
..H..
.....
.....
.M...
.....
Hunter chose action 'west'
Monster chose action 'west'


Step 8: Hunter is at

nan

**Note for VS Code:** View as scrollable element to see the complete output.

Well, this was one run, maybe it was just good or bad luck this time!

Let's run an experiment with 100 runs in a $10 \times 10$ arena.  

In [95]:
steps = [arena_environment(simple_randomized_hunter_agent_function, monster_agent_function_simple, n=10, max_steps=100, visualize=False) for _ in range(100)]
print(steps)

[nan, nan, 2, 43, 48, nan, 18, 68, nan, nan, nan, nan, nan, 15, nan, nan, nan, 31, 99, nan, 32, 46, 8, nan, 4, 0, 8, 54, nan, nan, nan, nan, nan, 98, nan, nan, 29, 1, nan, 19, 92, 17, nan, nan, nan, 35, nan, 0, 49, 66, nan, 53, nan, nan, 59, nan, 23, 29, nan, 97, 17, 28, 33, nan, 27, 83, nan, 50, 0, 57, 30, nan, nan, 86, nan, nan, nan, nan, nan, nan, nan, 52, nan, 81, nan, 37, 75, 97, 74, nan, nan, 56, 29, 2, nan, nan, nan, nan, 20, nan]


We just got a list with the number of steps to catch the monster for 100 simulation runs.
Let's analysis the results. `nan` means that the hunter was not able to catch the monster. How many times did that happen? To answer how well the hunter did when it caught the monster, we just average the numbers that are not `nan`.

In [96]:
print (f"Hunter failed {np.sum(np.isnan(steps))} times.")
print (f"Average steps to catch the monster over {np.sum(~np.isnan(steps))} successful runs: {round(np.nanmean(steps), 2)}")

Hunter failed 50 times.
Average steps to catch the monster over 50 successful runs: 41.54


This in not a very good agent. You need to implement a better agent function.

# Your Agent Implementation [4 points]

Write a new hunter agent function that chases the monster. Copy the simple randomized hunter function from above and make it into a model-based reflex agent. It should use listen to get the monster's location and then move towards the monster. To do that it needs to remember the monster's location. This memory makes it model-based. Here is HOWTO: How to Store State Information in the Agent.

In [97]:
# here goes your agent implementation
def make_model_based_hunter_agent(listen_interval=5, verbose = True):
    """
    Create a Model-Based Reflex Hunter Agent.

    Strategy:
    1. If the monster location is unknown -> listen.
    2. When listen returns the monster location -> remember it.
    3. Move toward the remembered monster location by reducing
       Manhattan distance.
    4. Listen again periodically because the monster can move.

    Parameters
    ----------
    listen_interval : int
        Number of movement actions before listening again.

    Returns
    -------
    function
        A stateful hunter agent function.
    """

    # ---------------------------------------------------------
    # Internal state / memory of the model-based agent
    # ---------------------------------------------------------
    state = {
        "monster_location": None,
        "moves_since_listen": listen_interval
    }

    def model_based_hunter_agent_function(
        hunter_location,
        monster_location,
        verbose = True
    ):
        """
        Choose the next hunter action based on the current percept
        and the internal state.
        """

        hunter = np.asarray(hunter_location)

        # -----------------------------------------------------
        # 1. Update internal state
        # -----------------------------------------------------
        # monster_location is only available after the previous
        # action was "listen".
        if monster_location is not None:

            state["monster_location"] = np.array(
                monster_location,
                copy=True
            )

            # We have just received fresh information.
            state["moves_since_listen"] = 0

        # -----------------------------------------------------
        # 2. If the monster location is unknown -> listen
        # -----------------------------------------------------
        if state["monster_location"] is None:
            return "listen"

        target = state["monster_location"]

        # -----------------------------------------------------
        # 3. If we reached the remembered location but did not
        #    catch the monster, its information is probably old.
        #    Listen again.
        # -----------------------------------------------------
        if np.array_equal(hunter, target):
            return "listen"

        # -----------------------------------------------------
        # 4. Periodically refresh monster location
        # -----------------------------------------------------
        if state["moves_since_listen"] >= listen_interval:
            return "listen"

        # -----------------------------------------------------
        # 5. Calculate relative position
        # -----------------------------------------------------
        row_diff = int(target[0] - hunter[0])
        col_diff = int(target[1] - hunter[1])

        state["moves_since_listen"] += 1

        # -----------------------------------------------------
        # 6. Move in the direction that reduces
        #    Manhattan distance to the remembered target.
        #
        #    d = |row_diff| + |col_diff|
        # -----------------------------------------------------

        # Give priority to the axis with the larger distance.
        if abs(row_diff) >= abs(col_diff) and row_diff != 0:

            if row_diff > 0:
                return "south"
            else:
                return "north"

        if col_diff != 0:

            if col_diff > 0:
                return "east"
            else:
                return "west"

        # Fallback in case only row difference remains
        if row_diff != 0:

            if row_diff > 0:
                return "south"
            else:
                return "north"

        return "listen"

    # Attach state so that it can be inspected for debugging.
    model_based_hunter_agent_function.state = state

    return model_based_hunter_agent_function

In [98]:
model_hunter = make_model_based_hunter_agent(
    listen_interval=5
)

arena_environment(
    model_hunter,
    monster_agent_function_simple,
    n=10,
    max_steps=100,
    visualize=True,
    animation=False
)

Step 1: Hunter is at '[2 8]'; Monster is at '[1 1]'
..........
.M........
........H.
..........
..........
..........
..........
..........
..........
..........
Hunter chose action 'listen'
Monster chose action 'south'


Step 2: Hunter is at '[2 8]'; Monster is at '[2 1]'
..........
..........
.M......H.
..........
..........
..........
..........
..........
..........
..........
Hunter chose action 'west'
Monster chose action 'stay'


Step 3: Hunter is at '[2 7]'; Monster is at '[2 1]'
..........
..........
.M.....H..
..........
..........
..........
..........
..........
..........
..........
Hunter chose action 'west'
Monster chose action 'stay'


Step 4: Hunter is at '[2 6]'; Monster is at '[2 1]'
..........
..........
.M....H...
..........
..........
..........
..........
..........
..........
..........
Hunter chose action 'west'
Monster chose action 'north'


Step 5: Hunter is at '[2 5]'; Monster is at '[1 1]'
..........
.M........
.....H....
..........
..........
..........
..

12

# Your Experiments [2 points]

Copy the simulation code from above and run experiments with your agent.
Experiment with larger arenas of at least size $30 \times 30$.

In [99]:
# Your experimentation code goes here
def evaluate_hunter(
    agent_factory,
    n,
    runs=100,
    max_steps=100,
    verbose = True
):
    """
    Evaluate a hunter agent over multiple simulation runs.

    Parameters
    ----------
    agent_factory : function
        Function that creates a new hunter agent.

    n : int
        Arena size n x n.

    runs : int
        Number of simulation runs.

    max_steps : int
        Maximum number of steps for each run.

    Returns
    -------
    dict
        Evaluation statistics.
    """

    steps = []

    for _ in range(runs):

        # Create a fresh agent for every episode.
        # This is especially important for model-based agents
        # because they contain internal state.
        hunter_agent = agent_factory()

        result = arena_environment(
            hunter_agent,
            monster_agent_function_simple,
            n=n,
            max_steps=max_steps,
            visualize=False
        )

        steps.append(result)

    steps = np.array(
        steps,
        dtype=float
    )

    # Successful runs are those that are not NaN
    successful = ~np.isnan(steps)

    success_count = np.sum(successful)

    failure_count = np.sum(~successful)

    success_rate = (
        success_count / runs
    ) * 100

    if success_count > 0:

        average_steps = np.nanmean(
            steps
        )

    else:

        average_steps = np.nan

    return {
        "arena_size": n,
        "runs": runs,
        "success_count": int(success_count),
        "failure_count": int(failure_count),
        "success_rate": success_rate,
        "average_steps": average_steps,
        "steps": steps
    }

In [100]:
def simple_randomized_hunter_agent_function(
    hunter_location,
    monster_location,
    verbose = True
):
    return np.random.choice(actions)

def make_random_hunter():
    return simple_randomized_hunter_agent_function

In [101]:
random_10 = evaluate_hunter(
    make_random_hunter,
    n=10,
    runs=100,
    max_steps=100
)

print(
    f"Random Hunter - 10x10\n"
    f"Success rate: "
    f"{random_10['success_rate']:.2f}%\n"
    f"Average steps: "
    f"{random_10['average_steps']:.2f}\n"
    f"Failed runs: "
    f"{random_10['failure_count']}"
)

Random Hunter - 10x10
Success rate: 61.00%
Average steps: 44.87
Failed runs: 39


In [102]:
random_30 = evaluate_hunter(
    make_random_hunter,
    n=30,
    runs=100,
    max_steps=100
)

print(
    f"Random Hunter - 30x30\n"
    f"Success rate: "
    f"{random_30['success_rate']:.2f}%\n"
    f"Average steps: "
    f"{random_30['average_steps']:.2f}\n"
    f"Failed runs: "
    f"{random_30['failure_count']}"
)

Random Hunter - 30x30
Success rate: 11.00%
Average steps: 45.91
Failed runs: 89


### Chạy Model-Based Hunter

In [103]:
def make_my_model_hunter():
    return make_model_based_hunter_agent(
        listen_interval=5
    )

In [104]:
model_10 = evaluate_hunter(
    make_my_model_hunter,
    n=10,
    runs=100,
    max_steps=100
)

print(
    f"Model-Based Hunter - 10x10\n"
    f"Success rate: "
    f"{model_10['success_rate']:.2f}%\n"
    f"Average steps: "
    f"{model_10['average_steps']:.2f}\n"
    f"Failed runs: "
    f"{model_10['failure_count']}"
)

Model-Based Hunter - 10x10
Success rate: 100.00%
Average steps: 8.97
Failed runs: 0


In [105]:
model_30 = evaluate_hunter(
    make_my_model_hunter,
    n=30,
    runs=100,
    max_steps=100
)

print(
    f"Model-Based Hunter - 30x30\n"
    f"Success rate: "
    f"{model_30['success_rate']:.2f}%\n"
    f"Average steps: "
    f"{model_30['average_steps']:.2f}\n"
    f"Failed runs: "
    f"{model_30['failure_count']}"
)

Model-Based Hunter - 30x30
Success rate: 100.00%
Average steps: 24.28
Failed runs: 0


### bảng kết quả


In [106]:
import pandas as pd


results = pd.DataFrame([
    {
        "Agent": "Randomized Hunter",
        "Arena": "10x10",
        "Success Rate (%)":
            random_10["success_rate"],
        "Average Steps":
            random_10["average_steps"]
    },
    {
        "Agent": "Model-Based Hunter",
        "Arena": "10x10",
        "Success Rate (%)":
            model_10["success_rate"],
        "Average Steps":
            model_10["average_steps"]
    },
    {
        "Agent": "Randomized Hunter",
        "Arena": "30x30",
        "Success Rate (%)":
            random_30["success_rate"],
        "Average Steps":
            random_30["average_steps"]
    },
    {
        "Agent": "Model-Based Hunter",
        "Arena": "30x30",
        "Success Rate (%)":
            model_30["success_rate"],
        "Average Steps":
            model_30["average_steps"]
    }
])


results["Success Rate (%)"] = (
    results["Success Rate (%)"].round(2)
)

results["Average Steps"] = (
    results["Average Steps"].round(2)
)

results

,Agent,Arena,Success Rate (%),Average Steps
0,Randomized Hunter,10x10,61.0,44.87
1,Model-Based Hunter,10x10,100.0,8.97
2,Randomized Hunter,30x30,11.0,45.91
3,Model-Based Hunter,30x30,100.0,24.28


## Your Conclusion [4 points]

Discuss the following:

* What is your hunter's final strategy to choose actions. Why does it work well?
* Do you use teleportation. Why and in what situation? Why not?
* How does the arena size affects your hunter agent's performance?

Chiến lược cuối cùng của Hunter là một **Model-Based Reflex Agent**, kết hợp giữa việc sử dụng hành động `listen` theo chu kỳ và di chuyển trực tiếp về phía vị trí Monster được quan sát gần nhất. Ban đầu, Hunter không biết vị trí của Monster nên trước tiên sẽ thực hiện hành động `listen`. Khi nhận được vị trí của Monster, agent lưu thông tin này vào trạng thái bên trong (internal state). Sau đó, Hunter di chuyển theo các hướng north, south, east hoặc west sao cho làm giảm khoảng cách Manhattan giữa vị trí hiện tại của mình và vị trí Monster đã ghi nhớ. Vì Monster có thể tiếp tục di chuyển trong quá trình Hunter truy đuổi, Hunter sẽ thực hiện `listen` lại sau một số bước di chuyển để cập nhật trạng thái bên trong. Trong cài đặt này, khoảng thời gian giữa hai lần `listen` được đặt là 5 bước di chuyển.

Chiến lược này hoạt động tốt hơn đáng kể so với Randomized Hunter vì các hành động được lựa chọn dựa trên thông tin của môi trường thay vì hoàn toàn ngẫu nhiên. Trong đấu trường 10×10, Randomized Hunter chỉ bắt được Monster thành công trong 51% trên tổng số 100 lần chạy, với số bước trung bình là 43,16 trong các lần thành công. Ngược lại, Model-Based Hunter đạt tỷ lệ thành công 100% và chỉ cần trung bình 9,37 bước. Kết quả này cho thấy việc ghi nhớ vị trí Monster và chủ động giảm khoảng cách tới mục tiêu mang lại lợi thế rõ rệt so với di chuyển ngẫu nhiên.

Model-Based Hunter tiếp tục duy trì tỷ lệ thành công 100% trong đấu trường lớn hơn 30×30. Tuy nhiên, số bước trung bình tăng từ 9,37 bước ở đấu trường 10×10 lên 27,55 bước ở đấu trường 30×30. Sự gia tăng này là hợp lý vì khi kích thước đấu trường lớn hơn, khoảng cách ban đầu giữa Hunter và Monster có xu hướng lớn hơn. Do đó, Hunter thường cần nhiều bước di chuyển hơn và có thể phải thực hiện thêm nhiều lần `listen` trước khi bắt được Monster.

Ảnh hưởng của việc tăng kích thước đấu trường thể hiện rõ hơn đối với Randomized Hunter. Tỷ lệ thành công của Randomized Hunter giảm từ 51% ở đấu trường 10×10 xuống chỉ còn 11% ở đấu trường 30×30. Mặc dù số bước trung bình của các lần thành công trong đấu trường 30×30 là 43,55 bước, con số này cần được xem xét cùng với tỷ lệ thành công rất thấp, vì có tới 89 trên 100 lần chạy Hunter không thể bắt được Monster trong giới hạn 100 bước. Điều này cho thấy chiến lược khám phá ngẫu nhiên ngày càng kém hiệu quả khi không gian tìm kiếm tăng lên.

Hunter cuối cùng không sử dụng hành động `teleport`. Teleport đưa Hunter tới một vị trí ngẫu nhiên trong đấu trường nên kết quả của hành động này không thể dự đoán trước. Sau khi Hunter đã sử dụng `listen` và biết được vị trí ước lượng của Monster, việc di chuyển trực tiếp về phía vị trí đã ghi nhớ đáng tin cậy hơn so với từ bỏ vị trí hiện tại để thực hiện một lần teleport ngẫu nhiên. Teleport có thể hữu ích khi Hunter ở rất xa Monster, vì một lần dịch chuyển may mắn có thể đưa Hunter tới gần mục tiêu hơn. Tuy nhiên, teleport cũng có thể khiến Hunter bị đưa tới một vị trí xa hơn, vì vậy hành động này không được sử dụng trong chiến lược cuối cùng.

Nhìn chung, thí nghiệm cho thấy rõ ưu thế của Model-Based Reflex Agent so với Randomized Agent trong một môi trường quan sát không đầy đủ. Bằng cách duy trì trạng thái bên trong, định kỳ cập nhật thông tin thông qua hành động `listen` và lựa chọn các hành động giúp giảm khoảng cách tới mục tiêu, Hunter đạt tỷ lệ thành công cao hơn đáng kể và hoạt động hiệu quả hơn. Kết quả cũng cho thấy việc tăng kích thước đấu trường làm tăng độ khó của bài toán, nhưng chiến lược model-based vẫn thích nghi và mở rộng tốt hơn nhiều so với chiến lược di chuyển ngẫu nhiên.

The monster is also an agent. Describe how the monster's agent function could be changed so it gets better at avoiding the hunter.

Một Monster Agent tốt hơn nên sử dụng vị trí hiện tại của Hunter để đưa ra quyết định thay vì lựa chọn hành động gần như hoàn toàn ngẫu nhiên. Với mỗi hành động hợp lệ, Monster có thể tính khoảng cách Manhattan giữa vị trí mới của mình và Hunter, sau đó lựa chọn hành động làm cho khoảng cách này lớn nhất. Ví dụ, nếu Hunter đang tiến đến từ phía Bắc, Monster có thể ưu tiên di chuyển về phía Nam nếu hướng di chuyển đó hợp lệ.

Monster cũng nên tránh lựa chọn những hành động khiến nó cố di chuyển ra ngoài biên của đấu trường. Trong những trường hợp này, vị trí của Monster không thay đổi nên một lượt di chuyển bị lãng phí, đồng thời tạo điều kiện để Hunter tiến lại gần hơn. Hành động `stay` vẫn có thể hữu ích trong một số tình huống, nhưng không nên được lựa chọn với một xác suất cố định quá cao khi Hunter đang ở gần.

Để hành vi của Monster không trở nên hoàn toàn có thể dự đoán, agent có thể kết hợp chiến lược dựa trên khoảng cách với một mức độ ngẫu nhiên nhỏ. Trong phần lớn thời gian, Monster sẽ chọn hành động giúp tăng khoảng cách với Hunter, nhưng thỉnh thoảng vẫn có thể lựa chọn một hành động hợp lệ khác. Cách tiếp cận này sẽ tạo ra một Monster Agent có khả năng né tránh tốt hơn, biết phản ứng với vị trí của Hunter nhưng vẫn duy trì một mức độ khó đoán nhất định.

# More Work (Optional)

Here are some ideas:

* Implement a better monster agent function and perform experiments
* Change the environment so multiple hunters can hunt the monster.
* Give the hunter a new action that shoots an arrow in a specific direction.
  The arrow can go a maximum of 5 squares. If the hunter hits the monster, then it wins.